# FAIRFIX — Exploratory Data Analysis & Model Prototyping
**ML-Based Price Intelligence and Valuation for Used and Out-of-Warranty Laptops**

This notebook documents the exploratory data analysis, feature engineering, and model prototyping for FairFix.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ml.data.preprocess import preprocess_and_save, get_feature_lists
from ml.predict import get_predictor

# Set plotting aesthetics
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
print("Imports loaded successfully.")

## 1. Dataset Ingestion & Feature Engineering
We load the engineered Indian Laptop Specifications Dataset from .

In [2]:
df = pd.read_csv("datasets/processed/laptops_featured.csv")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df[["brand", "device_category", "cpu_generation", "ram_gb", "storage_gb", "gpu_series", "price"]].head(5))

## 2. Target Variable Distribution (Price in INR)
Inspecting mean, median, skewness, and extreme values in the target price.

In [3]:
print(df["price"].describe())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["price"], bins=35, kde=True, ax=ax1, color="teal")
ax1.set_title("Laptop Price Distribution (INR ₹)")
ax1.set_xlabel("Price in INR")

sns.histplot(np.log1p(df["price"]), bins=35, kde=True, ax=ax2, color="purple")
ax2.set_title("Log-Transformed Price Distribution")
ax2.set_xlabel("Log(Price)")
plt.tight_layout()
plt.show()

## 3. Hardware Price Drivers: RAM, Storage & Dedicated GPU

In [4]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(x="ram_gb", y="price", data=df, ax=ax1, palette="Blues")
ax1.set_title("RAM Capacity vs Price (INR)")

sns.boxplot(x="is_dedicated_gpu", y="price", data=df, ax=ax2, palette="Set2")
ax2.set_title("Integrated (0) vs Dedicated GPU (1) vs Price")
plt.tight_layout()
plt.show()

## 4. Live Prediction & Valuation Verification

In [5]:
predictor = get_predictor()
sample_laptop = df.iloc[0].to_dict()
res = predictor.evaluate_valuation(sample_laptop, condition_dict={"is_used": True, "age_months": 12, "battery_health_percentage": 85}, asking_price=150000)
print("--- FairFix Valuation Output ---")
print(f"New Baseline: ₹{res["new_price_baseline_inr"]:,}")
print(f"Adjusted Second-Hand Value: ₹{res["valuation_inr"]:,}")
print("Deal Assessment:", res["deal_assessment"])